In [ ]:
import os
import sys
import pandas as pd
import torch
from torch.utils.data import DataLoader, Subset
from torchmetrics.image.inception import InceptionScore

sys.path.append(os.path.abspath("../"))
from src.data.dataset import INRDataset
from notebooks.generative import INRImageGenerator
from src.utils.prepare_data import INRDataProcessor
from src.models.gpt import INRGPT

class YamlConfig:
    def __init__(self, config_dict):
        for key, value in config_dict.items():
            setattr(self, key, value)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using target hardware acceleration: {device}")

if torch.cuda.is_available():
    torch.set_float32_matmul_precision("high")

CHECKPOINT_PATH = "../checkpoints/inr_gpt_final.pt"

if not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError(f"Missing checkpoint file at {CHECKPOINT_PATH}")

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
config = checkpoint.get("config")

print("Loading trained INRGPT engine...")
model = INRGPT(config).to(device)
print(vars(model.config))
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Loading dataset for structural template allocation...")
full_dataset = INRDataset(folder_path="../data/processed_inrs", split="train")

template_loader = DataLoader(
    Subset(full_dataset, range(64)),
    batch_size=64,
    shuffle=False,
    pin_memory=(device == "cuda"),
    num_workers=1,
)

reference_batch = next(iter(template_loader))
processor = INRDataProcessor(input_root="../data/mnist-inrs-relus/all_inrs_eight_neurons")
inception_score_metric = InceptionScore().to(device)

print(f"✅ Structural template locked. Sequence mapping layout shape: {reference_batch['layer_ids'][0].shape}")
def run_benchmark_experiment(
    mode, temperature, top_k, beam_width, beam_temperature=3.5, num_samples=1024
):
    """
    Executes an isolated evaluation pass using the static structural template.
    Diversity is driven entirely by open lottery selection at token step 1.
    """
    inception_score_metric.reset()

    # Instantiate the stateful generator with current hyperparameter constraints
    image_generator = INRImageGenerator(
        device=device,
        processor=processor,
        h=28,
        w=28,
        beam_width=beam_width,
        beam_temperature=beam_temperature,
        top_k=top_k,
        temperature=temperature,
    )

    is_batch_size = 64

    # Generate the target population pool fully in parallel chunks
    for _ in range(0, num_samples, is_batch_size):
        current_batch_size = min(is_batch_size, num_samples - _)

        # We pass the exact same structural layout template.
        # Higher temperatures and K values force the model to branch into different digits
        # immediately at token step 1 right after the identical start token!
        fake_images = image_generator.generate_images(
            gpt_model=model,
            num_samples=current_batch_size,
            batch_loader_sample=reference_batch,
            mode=mode,
        )

        inception_score_metric.update(fake_images)

    # Calculate final matrix metrics
    is_mean, is_std = inception_score_metric.compute()
    return is_mean.item(), is_std.item()

experiment_grid = [
    # --- 1. Greedy Argmax Path ---
    {"mode": "argmax", "temperature": 1.0, "top_k": 1, "beam_width": 1},
    # --- 2. Stochastic Top-K Sampling Configurations (Testing Diversity Fixes) ---
    {"mode": "sampling", "temperature": 1.0, "top_k": 9, "beam_width": 1},
    {"mode": "sampling", "temperature": 1.1, "top_k": 9, "beam_width": 1},
    {"mode": "sampling", "temperature": 1.25, "top_k": 15, "beam_width": 1},
    {"mode": "sampling", "temperature": 1.25, "top_k": 25, "beam_width": 1},
    {"mode": "sampling", "temperature": 1.35, "top_k": 50, "beam_width": 1},
    {"mode": "sampling", "temperature": 1.5, "top_k": 60, "beam_width": 1},
    {"mode": "sampling", "temperature": 1.7, "top_k": 70, "beam_width": 1},
    {"mode": "sampling", "temperature": 2.0, "top_k": 90, "beam_width": 1},
    {"mode": "sampling", "temperature": 2.2, "top_k": 100, "beam_width": 1},
    # --- 3. Structured Beam Search Paths ---
    # {"mode": "beam_search", "temperature": 1.0, "top_k": 1, "beam_width": 4},
    # {"mode": "beam_search", "temperature": 1.0, "top_k": 1, "beam_width": 8}
]

results_registry = []

print("🔬 Starting Hyperparameter Search Matrix Evaluation...\n")
print(f"{'MODE':<12} | {'TEMP':<5} | {'TOP-K':<5} | {'BEAM-TEMP':<9} | {'BEAM-W':<6} | {'INCEPTION SCORE':<18}")
print("-" * 69)

for exp in experiment_grid:
    try:
        mean, std = run_benchmark_experiment(
            mode=exp["mode"],
            temperature=exp["temperature"],
            top_k=exp["top_k"],
            beam_width=exp["beam_width"],
            num_samples=1024,
        )

        result_payload = {
            "Generation Mode": exp["mode"],
            "Temperature": exp["temperature"],
            "Top-K": exp["top_k"],
            "Beam Width": exp["beam_width"],
            "IS Mean": round(mean, 4),
            "IS Std": round(std, 4),
            "Display String": f"{mean:.4f} ± {std:.4f}",
        }

        results_registry.append(result_payload)

        print(f"{exp['mode']:<12} | {exp['temperature']:<5} | {exp['top_k']:<5} | {3.5:<9} | {exp['beam_width']:<6} | {result_payload['Display String']:<18}")

    except Exception as e:
        print(f"❌ Failed combo {exp['mode']} | Temp {exp['temperature']}: {str(e)}")

print("\n--- Final Sorting Matrix Results ---")
df = pd.DataFrame(results_registry)

# Sort by the highest mean Inception Score
leaderboard = df.sort_values(by="IS Mean", ascending=False).reset_index(drop=True)
print(leaderboard[["Generation Mode", "Temperature", "Top-K", "Beam Width", "Display String"]])